# 08.6 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/06-training-well/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow start-up messages
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import keras
from keras import layers
keras.utils.set_random_seed(42)      # same starting weights every run (small differences across machines are normal)
print("Keras", keras.__version__, "on", keras.backend.backend())

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]
cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)            # fit on training customers only (lesson 04.5)
Xs_train, Xs_test = scaler.transform(X_train), scaler.transform(X_test)

from sklearn.metrics import roc_auc_score
y_tr = y_train.to_numpy()
X_small, y_small = Xs_train[:300], y_tr[:300]        # a deliberately tiny training set
X_val, y_val = Xs_train[300:], y_tr[300:]            # validation: the other training customers

def small_net(learning_rate=0.001, optimizer=None):
    keras.utils.set_random_seed(42)
    m = keras.Sequential([keras.Input((5,)), layers.Dense(16, activation="relu"),
                          layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=optimizer or keras.optimizers.Adam(learning_rate), loss="binary_crossentropy")
    return m

def big_net(dropout=0.0, l2=0.0, batchnorm=False):
    """Two layers of 256 neurons: far too big for 300 customers, on purpose."""
    keras.utils.set_random_seed(42)
    reg = keras.regularizers.L2(l2) if l2 else None
    stack = [keras.Input((5,))]
    for _ in range(2):
        stack.append(layers.Dense(256, activation="relu", kernel_regularizer=reg))
        if batchnorm: stack.append(layers.BatchNormalization())
        if dropout: stack.append(layers.Dropout(dropout))
    m = keras.Sequential(stack + [layers.Dense(1, activation="sigmoid")])
    m.compile(optimizer=keras.optimizers.Adam(0.003), loss="binary_crossentropy")
    return m

def test_auc(m):
    return roc_auc_score(y_test, m.predict(Xs_test, verbose=0).ravel())

Keras 3.15.1 on tensorflow


3,317 customers, churn rate 43.6%, train 2,487 / test 830


## Exercise 1

Step 4: $u = 0.9(-0.271) - 0.1 = -0.3439$, $w = -0.9049$. Step 5: $u = -0.4095$, $w = -1.3144$. SGD is at −0.5: momentum has gone about 2.6 times as far.

In [2]:
w_sgd, w_mom, u = 0.0, 0.0, 0.0
for step in range(1, 6):
    w_sgd -= 0.1; u = 0.9 * u - 0.1; w_mom += u
print(f"after 5 steps: SGD {w_sgd:.3f}, momentum {w_mom:.4f}, ratio {w_mom / w_sgd:.1f}")

after 5 steps: SGD -0.500, momentum -1.3144, ratio 2.6


## Exercise 2

In [3]:
for lr in [3.0, 0.03]:
    h = small_net(lr).fit(Xs_train, y_train, epochs=30, batch_size=32, validation_split=0.2, verbose=0)
    print(f"lr {lr}: training loss epoch 1 / 10 / 30 = {h.history['loss'][0]:.3f} / "
          f"{h.history['loss'][9]:.3f} / {h.history['loss'][29]:.3f}, best val {min(h.history['val_loss']):.3f}")

lr 3.0: training loss epoch 1 / 10 / 30 = 11.872 / 0.727 / 0.699, best val 1.270


lr 0.03: training loss epoch 1 / 10 / 30 = 0.604 / 0.586 / 0.575, best val 0.585


3.0 behaves like 1.0 or worse: a huge first-epoch loss and no recovery. 0.03 sits with 0.01 and 0.1 in the range that works.

## Exercise 3

In [4]:
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
m = big_net(dropout=0.5)
h = m.fit(X_small, y_small, epochs=200, batch_size=32, validation_data=(X_val, y_val), callbacks=[es], verbose=0)
print(f"dropout + early stopping: {len(h.history['loss'])} epochs, test AUC {test_auc(m):.3f}")

dropout + early stopping: 14 epochs, test AUC 0.731


About the same as early stopping alone (0.730) and well above dropout alone (0.684). Dropout slows memorising, so early stopping has a wider window to stop in; on bigger problems the combination is the usual practice, here the gain is negligible.

## Exercise 4

In [5]:
models_dir = data_path("download.py").parent.parent / "models"
models_dir.mkdir(exist_ok=True)
path = models_dir / "churn_checkpoint.keras"
ckpt = keras.callbacks.ModelCheckpoint(path, monitor="val_loss", save_best_only=True)
big_net().fit(X_small, y_small, epochs=60, batch_size=32, validation_data=(X_val, y_val), callbacks=[ckpt], verbose=0)
best = keras.models.load_model(path)
print(f"best checkpoint, test AUC {test_auc(best):.3f}")

best checkpoint, test AUC 0.730


The checkpoint holds the best epoch even though training ran on and overfitted. In long training runs (hours, on images) checkpoints also protect you from crashes.